In [21]:
import cv2
import fiona
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import rasterio
from rasterio.windows import Window
from rasterio.stack import stack
from rasterio.enums import Resampling
from rasterio.plot import show
import rasterio.mask
import tempfile
import tqdm

In [ ]:
folder_path = "../data/"
RGB_path = "20250116_RGB.tif"
RGB_clip =  "20250116_RGB_clip.tif"
RGB_resize =  "20250116_RGB_clip_resize.tif"
tile_folder = os.path.join(folder_path, "tiles")
os.makedirs(tile_folder, exist_ok=True)
tile_size = 1024
overlap = 200
stride = tile_size - overlap
border_pixel = 50

In [3]:
for file in os.listdir(folder_path):
    if file.endswith("B02_10m.jp2"):
       B2 = os.path.join(folder_path, file)
        
    if file.endswith("B03_10m.jp2"):
       B3 = os.path.join(folder_path, file)

    if file.endswith("B04_10m.jp2"):
       B4 = os.path.join(folder_path, file)

sources = [B4, B3, B2]

dst_kwds = {
    "driver": "GTiff",
    "compress": "deflate",
    "tiled": True,
    "BIGTIFF": "IF_SAFER",
}

stack(
    sources=sources,
    dst_path= os.path.join(folder_path, RGB_path),
    dst_kwds=dst_kwds,
    resampling=Resampling.nearest,
)

In [6]:
# clip to AOI
with fiona.open("data/AOI.shp", "r") as shapefile:
        shape = [feature["geometry"] for feature in shapefile]

with rasterio.open(os.path.join(folder_path, RGB_path)) as src:
    out_img, out_transform = rasterio.mask.mask(src, shape, crop=True)
    out_meta = src.meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": out_img.shape[1],
        "width": out_img.shape[2],
        "transform": out_transform
    })

    with rasterio.open(os.path.join(folder_path, RGB_clip) , "w", **out_meta) as dest:
        dest.write(out_img)

In [10]:
with rasterio.open(os.path.join(folder_path, RGB_clip)) as src: 
    data = src.read()
    transform = src.transform
    crs = src.crs

data_cv = data.transpose(1,2,0)

new_height = data_cv.shape[0] * 5
new_width = data_cv.shape[1] * 5

resized = cv2.resize(data_cv, (new_width, new_height), interpolation = cv2.INTER_LANCZOS4)
resized = resized.transpose(2, 0, 1)

#change transformation for new resolution 
new_transform = transform * rasterio.transform.Affine.scale(data_cv.shape[1] / new_width, data_cv.shape[0] / new_height)

with rasterio.open(os.path.join(folder_path, RGB_resize),
                   "w", driver="Gtiff",
                   height = new_height,
                   width=new_width,
                   count=resized.shape[0],
                   dtype = resized.dtype,
                   crs=crs,
                   transform=new_transform
                  ) as dst:
    dst.write(resized)

In [25]:
%%time
#Generate tiles and perform segmentation
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    
    with rasterio.open(os.path.join(folder_path, RGB_resize)) as dataset:
        meta_base = dataset.meta.copy()
        tile_id = 0
        full_image = dataset.read()
        global_max = full_image.max()        

        for x in range(0, dataset.width, stride):
            for y in range(0, dataset.height, stride): 
                window = Window(x, y, tile_size, tile_size)
                dn = dataset.read(window=window)
        
                #if tiles are smaller than others, only shift as much as needed to reach tilesize (bigger overlap)
                if dn.shape[1] < tile_size or dn.shape[2] < tile_size:
                    offset_x = tile_size - dn.shape[2]
                    offset_y = tile_size - dn.shape[1]
                    x = x - offset_x
                    y = y - offset_y
                    window = Window(x, y, tile_size, tile_size)
                    dn = dataset.read(window=window)
        
                if dn.sum() == 0:
                    continue
                win_transform = dataset.window_transform(window)
                win_meta = dataset.meta
                win_meta.update(
                    {
                        "height": tile_size,
                        "width": tile_size,
                        "transform": win_transform,
                    })
                output_path = os.path.join(tmp, f"tile_{tile_id:03d}_x{x}_y{y}.tif")
        
                with rasterio.open(output_path, "w", **win_meta) as dst:
                    dst.write(dn.astype(np.float32))
        
                tile_id += 1
        
        tiles = glob.glob(str(tmp / "*.tif"))

        for t in tqdm.tqdm(tiles):
            with rasterio.open(t) as dataset:
                image = dataset.read()
                transform = dataset.transform
                crs = dataset.crs
                rgb = dataset.read([1, 2, 3])
                   
                tile_num = int(Path(t).stem.split('_')[1])
                rgb = np.clip(rgb / global_max, 0, 1) 
                
                rgb_255 = (rgb * 255).astype(np.uint8)

                bands_clahe = []
                for band in rgb_255: 
                     clahe = cv2.createCLAHE(clipLimit = 2)
                     band_clahe = clahe.apply(band)
                     bands_clahe.append(band_clahe)
                rgb_255_clahe = np.stack((bands_clahe[0], bands_clahe[1], bands_clahe[2]), axis = -1) 
                bgr_255 = cv2.cvtColor(rgb_255_clahe, cv2.COLOR_RGB2BGR)
                

                jpg_path = os.path.join(tile_folder, f"tile_{tile_num}.jpg")
                cv2.imwrite(jpg_path, bgr_255, [cv2.IMWRITE_JPEG_QUALITY, 100])

100%|█████████████████████████████████████████| 576/576 [00:14<00:00, 40.06it/s]


CPU times: user 40.8 s, sys: 13.6 s, total: 54.4 s
Wall time: 24.5 s
